# Chapter 8.4: Multi-Modal Model Integration

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** multi-modal model architecture patterns
2. **Implement** image processor integration
3. **Build** a vision encoder
4. **Compare** cross-attention vs fused input approaches
5. **Use** vLLM's multi-modal input handling system
6. **Implement** the `SupportsMultiModal` interface
7. **Walk through** the LLaVA implementation in vLLM
8. **Add** image support to a text-only model

---

## Prerequisites
- Chapter 8.1-8.2 (Model Registration and Implementation)
- Basic understanding of vision transformers (ViT)
- Familiarity with CLIP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.4_multimodal.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.4_multimodal.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Multi-Modal Architecture Patterns

There are several approaches to combining vision and language:

```
Pattern 1: Fused Input (LLaVA-style)
+-------+     +----------+     +-----------+
| Image | --> | Vision   | --> | Projector | --> [image_tokens]
+-------+     | Encoder  |     +-----------+        |
              +----------+                     concatenate
+-------+                                           |
| Text  | --> [text_tokens] ----------------------> [combined_tokens]
+-------+                                           |
                                              +-----------+
                                              |  Language  |
                                              |   Model    |
                                              +-----------+

Pattern 2: Cross-Attention (Flamingo-style)
+-------+     +----------+     +-----------+
| Image | --> | Vision   | --> | Perceiver | --> vision_features
+-------+     | Encoder  |     | Resampler |        |
              +----------+     +-----------+   cross-attend
+-------+                                           |
| Text  | --> [text_tokens] --> Language Model <-----+
+-------+                     (with cross-attn layers)

Pattern 3: Early Fusion (Fuyu-style)
+-------+     +----------+
| Image | --> | Patchify | --> [patch_embeddings]
+-------+     +----------+          |
                              concatenate
+-------+                           |
| Text  | --> [text_embeddings] --> [combined_embeddings]
+-------+                           |
                              +-----------+
                              |  Language  |
                              |   Model    |
                              +-----------+
```

In [ ]:
import torch
import torch.nn as nn
from typing import Optional, List, Tuple, Dict, Any

# Summary of multi-modal patterns
patterns = {
    "Fused Input (LLaVA)": {
        "models": ["LLaVA", "LLaVA-NeXT", "InternVL", "Qwen-VL"],
        "approach": "Vision features are projected to LLM embedding space and concatenated with text tokens",
        "pros": ["Simple, reuses existing LLM", "No architectural changes to LLM"],
        "cons": ["Long sequence length (image = many tokens)", "Image tokens use KV cache"],
    },
    "Cross-Attention (Flamingo)": {
        "models": ["Flamingo", "IDEFICS", "Emu"],
        "approach": "Vision features attend to text via cross-attention layers inserted in LLM",
        "pros": ["Efficient (fewer image tokens)", "Better scaling"],
        "cons": ["Requires modifying LLM architecture", "More complex implementation"],
    },
    "Early Fusion (Fuyu)": {
        "models": ["Fuyu", "PaLI"],
        "approach": "Image patches directly used as input tokens (no separate vision encoder)",
        "pros": ["Simplest architecture", "Supports arbitrary resolution"],
        "cons": ["Very long sequences", "Needs lots of training data"],
    },
}

for name, info in patterns.items():
    print(f"\n{name}")
    print(f"  Models: {', '.join(info['models'])}")
    print(f"  Approach: {info['approach']}")
    print(f"  Pros: {info['pros']}")
    print(f"  Cons: {info['cons']}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

def draw_multimodal_architecture():
    """Multi-modal architecture diagram: Image -> ViT -> Projection -> LLM -> Output."""
    fig, ax = plt.subplots(figsize=(18, 7))
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 8)
    ax.axis('off')
    ax.set_title('Multi-Modal Architecture (LLaVA-style Fused Input)', fontsize=16, fontweight='bold', pad=15)

    # --- Image Path (top) ---
    components_top = [
        (1.0, 5.5, 2.2, 1.5, 'Image\nInput', '#4A90D9'),
        (4.5, 5.5, 2.8, 1.5, 'Vision Encoder\n(ViT / CLIP)', '#27AE60'),
        (8.5, 5.5, 2.5, 1.5, 'Projection\n(MLP 2-layer)', '#F39C12'),
    ]
    for x, y, w, h, label, color in components_top:
        rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                              facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(rect)
        ax.text(x + w / 2, y + h / 2, label, ha='center', va='center',
                fontsize=10, fontweight='bold', color='white')

    # Arrows on top path
    for x1, x2 in [(3.2, 4.5), (7.3, 8.5)]:
        ax.annotate('', xy=(x2, 6.25), xytext=(x1, 6.25),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2))

    # Image tokens label
    ax.text(11.3, 6.25, '[Image Tokens]', ha='left', va='center', fontsize=10,
            fontweight='bold', color='#F39C12',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0', edgecolor='#F39C12'))

    # --- Text Path (bottom) ---
    text_box = FancyBboxPatch((1.0, 2.0), 2.2, 1.5, boxstyle='round,pad=0.12',
                              facecolor='#4A90D9', edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(text_box)
    ax.text(2.1, 2.75, 'Text\nInput', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

    ax.text(4.5, 2.75, '[Text Tokens]', ha='left', va='center', fontsize=10,
            fontweight='bold', color='#4A90D9',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E3F2FD', edgecolor='#4A90D9'))

    # Arrow from text
    ax.annotate('', xy=(4.4, 2.75), xytext=(3.2, 2.75),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

    # --- Concatenation point ---
    concat_x, concat_y = 12.5, 4.25
    circle = plt.Circle((concat_x, concat_y), 0.5, facecolor='#E74C3C', edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(circle)
    ax.text(concat_x, concat_y, '+', ha='center', va='center', fontsize=18, fontweight='bold', color='white')
    ax.text(concat_x, concat_y - 0.75, 'Concatenate', ha='center', fontsize=8, color='#E74C3C', fontweight='bold')

    # Arrows into concat
    ax.annotate('', xy=(concat_x - 0.5, concat_y + 0.3), xytext=(11.0 + 1.7, 6.25),
                arrowprops=dict(arrowstyle='->', color='#F39C12', lw=2, connectionstyle='arc3,rad=-0.2'))
    ax.annotate('', xy=(concat_x - 0.5, concat_y - 0.3), xytext=(6.5, 2.75),
                arrowprops=dict(arrowstyle='->', color='#4A90D9', lw=2, connectionstyle='arc3,rad=0.2'))

    # --- LLM ---
    llm_box = FancyBboxPatch((14.0, 3.25), 2.5, 2.0, boxstyle='round,pad=0.15',
                             facecolor='#8E44AD', edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(llm_box)
    ax.text(15.25, 4.25, 'LLM\n(Decoder-Only\nTransformer)', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

    ax.annotate('', xy=(14.0, 4.25), xytext=(concat_x + 0.5, concat_y),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

    # --- Output ---
    out_box = FancyBboxPatch((17.3, 3.5), 2.0, 1.5, boxstyle='round,pad=0.12',
                             facecolor='#27AE60', edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(out_box)
    ax.text(18.3, 4.25, 'Output\nText', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

    ax.annotate('', xy=(17.3, 4.25), xytext=(16.5, 4.25),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

    plt.tight_layout()
    plt.savefig('multimodal_architecture.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_multimodal_architecture()

## 2. Vision Encoder: CLIP ViT

Most multi-modal models use a CLIP vision encoder. Let's implement a simplified version.

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert image to patch embeddings."""
    
    def __init__(self, image_size=224, patch_size=14, in_channels=3, embed_dim=1024):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2  # 256 for 224/14
        
        # Conv2d acts as patch embedding
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, channels, height, width]
        # Output: [batch, num_patches, embed_dim]
        x = self.proj(x)  # [B, embed_dim, H/P, W/P]
        x = x.flatten(2).transpose(1, 2)  # [B, num_patches, embed_dim]
        return x


class VisionTransformerBlock(nn.Module):
    """Simplified ViT transformer block."""
    
    def __init__(self, embed_dim=1024, num_heads=16, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
        )
    
    def forward(self, x):
        # Pre-norm transformer block
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x


class SimplifiedCLIPVisionModel(nn.Module):
    """
    Simplified CLIP Vision Encoder.
    
    In real vLLM, this would be loaded from HuggingFace's CLIPVisionModel.
    """
    
    def __init__(
        self,
        image_size: int = 224,
        patch_size: int = 14,
        embed_dim: int = 1024,
        num_layers: int = 24,
        num_heads: int = 16,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_patches = (image_size // patch_size) ** 2
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(
            image_size, patch_size, 3, embed_dim
        )
        
        # CLS token and position embedding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, embed_dim)
        )
        
        # Transformer blocks
        self.layers = nn.ModuleList([
            VisionTransformerBlock(embed_dim, num_heads)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pixel_values: [batch, channels, height, width]
        Returns:
            image_features: [batch, num_patches+1, embed_dim]
        """
        batch_size = pixel_values.shape[0]
        
        # Get patch embeddings
        x = self.patch_embed(pixel_values)  # [B, num_patches, embed_dim]
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # [B, num_patches+1, embed_dim]
        
        # Add position embeddings
        x = x + self.pos_embed
        
        # Process through transformer
        for layer in self.layers:
            x = layer(x)
        
        x = self.norm(x)
        return x

# Test vision encoder
vision_encoder = SimplifiedCLIPVisionModel(
    image_size=224, patch_size=14, embed_dim=256,
    num_layers=2, num_heads=4  # Small for testing
)

# Simulate an image
fake_image = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    features = vision_encoder(fake_image)

print(f"Image input: {fake_image.shape}")
print(f"Vision features: {features.shape}")
print(f"  = 1 CLS token + {(224//14)**2} patch tokens")
print(f"  = {features.shape[1]} tokens x {features.shape[2]} dim")
print(f"Vision encoder params: {sum(p.numel() for p in vision_encoder.parameters()):,}")

## 3. Multi-Modal Projector

The projector maps vision features to the language model's embedding space.

In [ ]:
class MultiModalProjector(nn.Module):
    """
    Projects vision features to LLM embedding space.
    
    Different models use different projector designs:
    - LLaVA: 2-layer MLP
    - LLaVA-NeXT: 2-layer MLP with GELU
    - InternVL: QLLaMA projector
    - Qwen-VL: Perceiver Resampler
    """
    
    def __init__(
        self,
        vision_dim: int,   # Vision encoder output dim
        text_dim: int,     # LLM hidden dim
        projector_type: str = "mlp2x",
    ):
        super().__init__()
        self.projector_type = projector_type
        
        if projector_type == "linear":
            self.proj = nn.Linear(vision_dim, text_dim)
        
        elif projector_type == "mlp2x":
            # LLaVA-style: 2-layer MLP
            self.proj = nn.Sequential(
                nn.Linear(vision_dim, text_dim),
                nn.GELU(),
                nn.Linear(text_dim, text_dim),
            )
        
        elif projector_type == "perceiver":
            # Perceiver Resampler: reduces number of tokens
            self.num_queries = 64  # Fixed number of output tokens
            self.queries = nn.Parameter(torch.randn(1, self.num_queries, text_dim))
            self.cross_attn = nn.MultiheadAttention(
                text_dim, num_heads=8, batch_first=True
            )
            self.input_proj = nn.Linear(vision_dim, text_dim)
            self.norm = nn.LayerNorm(text_dim)
        
        else:
            raise ValueError(f"Unknown projector type: {projector_type}")
    
    def forward(self, vision_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            vision_features: [batch, num_patches, vision_dim]
        Returns:
            projected: [batch, num_tokens, text_dim]
        """
        if self.projector_type in ["linear", "mlp2x"]:
            return self.proj(vision_features)
        
        elif self.projector_type == "perceiver":
            batch_size = vision_features.shape[0]
            kv = self.input_proj(vision_features)
            queries = self.queries.expand(batch_size, -1, -1)
            output, _ = self.cross_attn(queries, kv, kv)
            return self.norm(output)

# Compare projector types
vision_dim = 256
text_dim = 512
num_patches = 257  # 256 patches + 1 CLS
vision_features = torch.randn(1, num_patches, vision_dim)

for proj_type in ["linear", "mlp2x", "perceiver"]:
    proj = MultiModalProjector(vision_dim, text_dim, proj_type)
    with torch.no_grad():
        out = proj(vision_features)
    print(f"\n{proj_type}:")
    print(f"  Input:  {vision_features.shape} ({num_patches} tokens)")
    print(f"  Output: {out.shape} ({out.shape[1]} tokens)")
    print(f"  Params: {sum(p.numel() for p in proj.parameters()):,}")

## 4. Image Processing Pipeline

Before images reach the vision encoder, they need to be preprocessed.

In [ ]:
class SimpleImageProcessor:
    """
    Image preprocessing pipeline.
    
    In real vLLM, this uses HuggingFace's image processor:
    AutoImageProcessor.from_pretrained(model_name)
    """
    
    def __init__(
        self,
        image_size: int = 224,
        mean: Tuple[float, ...] = (0.48145466, 0.4578275, 0.40821073),  # CLIP mean
        std: Tuple[float, ...] = (0.26862954, 0.26130258, 0.27577711),  # CLIP std
    ):
        self.image_size = image_size
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std = torch.tensor(std).view(3, 1, 1)
    
    def preprocess(self, images: List[torch.Tensor]) -> torch.Tensor:
        """
        Preprocess a list of images.
        
        Steps:
        1. Resize to image_size x image_size
        2. Normalize pixel values to [0, 1]
        3. Apply CLIP normalization (mean/std)
        """
        processed = []
        for img in images:
            # Assume img is [C, H, W] float tensor in [0, 1]
            if img.shape[1] != self.image_size or img.shape[2] != self.image_size:
                img = torch.nn.functional.interpolate(
                    img.unsqueeze(0), size=(self.image_size, self.image_size),
                    mode='bilinear', align_corners=False
                ).squeeze(0)
            
            # Normalize
            img = (img - self.mean) / self.std
            processed.append(img)
        
        return torch.stack(processed)  # [batch, C, H, W]

# Test image processing
processor = SimpleImageProcessor(image_size=224)

# Simulate images of different sizes
images = [
    torch.rand(3, 320, 480),  # landscape
    torch.rand(3, 480, 320),  # portrait
    torch.rand(3, 224, 224),  # already correct size
]

pixel_values = processor.preprocess(images)
print(f"Preprocessed images: {pixel_values.shape}")
print(f"Value range: [{pixel_values.min():.2f}, {pixel_values.max():.2f}]")
print(f"Mean per channel: {pixel_values.mean(dim=(0, 2, 3)).tolist()}")

## 5. vLLM Multi-Modal Input Handling

vLLM has a dedicated system for handling multi-modal inputs. Let's understand how it works.

In [ ]:
# vLLM Multi-Modal Input System

print("vLLM Multi-Modal Input Pipeline")
print("=" * 60)
print("""
User Input:
    prompt = "<image>\nDescribe this image."
    images = [PIL.Image]  or  pixel_values = [tensor]

Step 1: Input Processing (MultiModalInputMapper)
    - Tokenize text prompt
    - Process images through image processor
    - Create MultiModalInputs dict:
      {
          "pixel_values": tensor [B, C, H, W],
          "image_sizes": [(H, W), ...],  # original sizes
      }

Step 2: Placeholder Token Handling
    - Count <image> tokens in the prompt
    - Replace each <image> with N placeholder tokens
    - N = number of vision tokens per image

Step 3: Embedding Merge (in model.forward())
    - Compute text embeddings for all tokens
    - Compute image embeddings via vision encoder + projector
    - Replace placeholder token embeddings with image embeddings
    - Feed merged embeddings to the LLM

Step 4: KV Cache
    - Image tokens are stored in KV cache like text tokens
    - During decoding, image context is available via cached KV
""")

In [ ]:
# The SupportsMultiModal interface

from typing import Protocol, runtime_checkable

@runtime_checkable
class SupportsMultiModal(Protocol):
    """
    Protocol for multi-modal models.
    
    Models implementing this interface can accept image/video inputs.
    File: vllm/model_executor/models/interfaces.py
    """
    pass

# In vLLM, multi-modal support is declared via:
# 1. The SupportsMultiModal protocol
# 2. The MULTIMODAL_REGISTRY decorator
# 3. An input_mapper and input_processor

MULTIMODAL_REGISTRY_CODE = '''
# How to register a multi-modal model in vLLM

from vllm.multimodal import MULTIMODAL_REGISTRY

@MULTIMODAL_REGISTRY.register_image_input_mapper()  # Register image handling
@MULTIMODAL_REGISTRY.register_max_image_tokens(get_max_image_tokens)  # Token budget
class MyVisionLanguageModel(nn.Module, SupportsMultiModal):
    """
    Multi-modal model with image support.
    
    The decorators tell vLLM:
    1. How to process image inputs for this model
    2. How many tokens each image will produce
    """
    pass
'''

print(MULTIMODAL_REGISTRY_CODE)

## 6. LLaVA Architecture Walkthrough

LLaVA (Large Language and Vision Assistant) is the most common multi-modal architecture. Let's walk through its implementation.

In [ ]:
class SimplifiedLLaVA(nn.Module):
    """
    Simplified LLaVA implementation showing the key concepts.
    
    Architecture:
    1. Vision Encoder (CLIP ViT) - processes images
    2. Multi-Modal Projector (MLP) - maps vision to text space
    3. Language Model (Llama) - generates text
    
    The key insight: image tokens are treated just like text tokens
    after projection. The LLM doesn't know they came from an image!
    """
    
    # Special token IDs
    IMAGE_TOKEN_ID = 32000  # <image> token
    
    def __init__(self, config):
        super().__init__()
        
        # Vision encoder (frozen CLIP)
        self.vision_tower = SimplifiedCLIPVisionModel(
            image_size=config.image_size,
            patch_size=config.patch_size,
            embed_dim=config.vision_dim,
            num_layers=2,  # Small for demo
            num_heads=4,
        )
        
        # Multi-modal projector
        self.multi_modal_projector = MultiModalProjector(
            vision_dim=config.vision_dim,
            text_dim=config.hidden_size,
            projector_type="mlp2x",
        )
        
        # Language model components
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=config.hidden_size,
                nhead=config.num_heads,
                dim_feedforward=config.intermediate_size,
                batch_first=True,
            )
            for _ in range(config.num_layers)
        ])
        self.norm = nn.LayerNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        
        self.config = config
    
    def _get_image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Process images through vision encoder + projector.
        
        Returns image features in the LLM's embedding space.
        """
        # Step 1: Vision encoder
        vision_features = self.vision_tower(pixel_values)
        # Remove CLS token (LLaVA uses patch tokens only)
        vision_features = vision_features[:, 1:, :]  # [B, num_patches, vision_dim]
        
        # Step 2: Project to LLM space
        image_features = self.multi_modal_projector(vision_features)
        # [B, num_patches, hidden_size]
        
        return image_features
    
    def _merge_image_and_text(
        self,
        input_ids: torch.Tensor,         # [batch, seq_len]
        image_features: torch.Tensor,     # [num_images, num_patches, hidden]
    ) -> torch.Tensor:
        """
        Merge image features into the text embedding sequence.
        
        Replace <image> token embeddings with actual image features.
        """
        # Get text embeddings
        text_embeddings = self.embed_tokens(input_ids)  # [B, seq_len, hidden]
        
        # Find <image> token positions
        image_mask = input_ids == self.IMAGE_TOKEN_ID
        
        # For simplicity, handle one image per sequence
        batch_size = input_ids.shape[0]
        for b in range(batch_size):
            image_positions = torch.where(image_mask[b])[0]
            if len(image_positions) > 0 and b < image_features.shape[0]:
                # Replace placeholder embeddings with image features
                num_image_tokens = min(len(image_positions), image_features.shape[1])
                for i in range(num_image_tokens):
                    pos = image_positions[i]
                    text_embeddings[b, pos] = image_features[b, i]
        
        return text_embeddings
    
    def forward(
        self,
        input_ids: torch.Tensor,
        pixel_values: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Forward pass for LLaVA.
        
        Args:
            input_ids: Token IDs including <image> placeholders
            pixel_values: Preprocessed images [B, C, H, W]
        """
        # Process images if provided
        if pixel_values is not None:
            image_features = self._get_image_features(pixel_values)
            inputs_embeds = self._merge_image_and_text(input_ids, image_features)
        else:
            inputs_embeds = self.embed_tokens(input_ids)
        
        # Process through LLM
        hidden_states = inputs_embeds
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        
        hidden_states = self.norm(hidden_states)
        logits = self.lm_head(hidden_states)
        
        return logits

# Test LLaVA
class LLaVAConfig:
    vocab_size = 32001  # +1 for <image> token
    hidden_size = 256
    intermediate_size = 512
    num_heads = 4
    num_layers = 2
    image_size = 224
    patch_size = 14
    vision_dim = 256

config = LLaVAConfig()
model = SimplifiedLLaVA(config)

# Create input with image placeholder
# Token sequence: [BOS, <image>, <image>, ..., text_tokens]
num_image_tokens = (224 // 14) ** 2  # 256 patches
input_ids = torch.cat([
    torch.tensor([[1]]),  # BOS
    torch.full((1, num_image_tokens), 32000),  # <image> placeholders
    torch.randint(1, 32000, (1, 10)),  # text tokens
], dim=1)

pixel_values = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    logits = model(input_ids, pixel_values)

print(f"Input tokens: {input_ids.shape[1]} (1 BOS + {num_image_tokens} image + 10 text)")
print(f"Image: {pixel_values.shape}")
print(f"Output logits: {logits.shape}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

## 7. vLLM's Actual LLaVA Implementation

Let's look at how vLLM implements LLaVA in its actual source code.

In [ ]:
VLLM_LLAVA_CODE = '''
# File: vllm/model_executor/models/llava.py (simplified)

from vllm.multimodal import MULTIMODAL_REGISTRY
from vllm.model_executor.models.interfaces import SupportsMultiModal

def get_max_llava_image_tokens(ctx):
    """Calculate max image tokens for memory allocation."""
    hf_config = ctx.get_hf_config(LlavaConfig)
    vision_config = hf_config.vision_config
    
    image_size = vision_config.image_size
    patch_size = vision_config.patch_size
    num_patches = (image_size // patch_size) ** 2
    
    return num_patches  # Each patch becomes one token

def input_processor_for_llava(ctx, llm_inputs):
    """Process inputs: expand <image> placeholder to correct number of tokens."""
    multi_modal_data = llm_inputs.get("multi_modal_data")
    if multi_modal_data is None:
        return llm_inputs
    
    # Replace single <image> token with N placeholder tokens
    num_image_tokens = get_max_llava_image_tokens(ctx)
    prompt_token_ids = llm_inputs["prompt_token_ids"]
    
    # Find and expand image placeholders
    new_token_ids = []
    for token_id in prompt_token_ids:
        if token_id == IMAGE_TOKEN_INDEX:
            new_token_ids.extend([IMAGE_TOKEN_INDEX] * num_image_tokens)
        else:
            new_token_ids.append(token_id)
    
    llm_inputs["prompt_token_ids"] = new_token_ids
    return llm_inputs


@MULTIMODAL_REGISTRY.register_image_input_mapper()
@MULTIMODAL_REGISTRY.register_max_image_tokens(get_max_llava_image_tokens)
@INPUT_REGISTRY.register_input_processor(input_processor_for_llava)
class LlavaForConditionalGeneration(nn.Module, SupportsMultiModal):
    """
    LLaVA model for vLLM.
    """
    
    def __init__(self, config, multimodal_config, cache_config, quant_config):
        super().__init__()
        
        # Vision encoder (CLIP)
        self.vision_tower = CLIPVisionModel(config.vision_config)
        
        # Projector
        self.multi_modal_projector = LlavaMultiModalProjector(
            config.vision_config.hidden_size,
            config.text_config.hidden_size,
            projector_hidden_act=config.projector_hidden_act,
        )
        
        # Language model (e.g., Llama)
        self.language_model = init_vllm_registered_model(
            config.text_config, cache_config, quant_config
        )
    
    def _process_image_input(self, image_input):
        """Process image input dict to get image features."""
        pixel_values = image_input["pixel_values"]
        vision_features = self.vision_tower(pixel_values)
        image_features = self.multi_modal_projector(vision_features)
        return image_features
    
    def forward(
        self,
        input_ids,
        positions,
        kv_caches,
        attn_metadata,
        intermediate_tensors=None,
        **kwargs,  # Contains pixel_values
    ):
        image_input = self._parse_and_validate_image_input(**kwargs)
        
        if image_input is not None:
            image_features = self._process_image_input(image_input)
            inputs_embeds = self.language_model.model.embed_tokens(input_ids)
            inputs_embeds = merge_multimodal_embeddings(
                input_ids, inputs_embeds, image_features, IMAGE_TOKEN_INDEX
            )
            input_ids = None  # Use inputs_embeds instead
        else:
            inputs_embeds = None
        
        hidden_states = self.language_model.model(
            input_ids, positions, kv_caches, attn_metadata,
            inputs_embeds=inputs_embeds,
        )
        return hidden_states
'''

print(VLLM_LLAVA_CODE)

## 8. The `merge_multimodal_embeddings` Function

In [ ]:
def merge_multimodal_embeddings(
    input_ids: torch.Tensor,       # [num_tokens]
    text_embeds: torch.Tensor,     # [num_tokens, hidden]
    image_features: torch.Tensor,  # [num_images, num_patches, hidden]
    image_token_id: int,           # Token ID for <image>
) -> torch.Tensor:
    """
    Replace image placeholder embeddings with actual image features.
    
    This is the key function that merges vision and text!
    
    Before:
        text_embeds: [BOS_emb, <img>_emb, <img>_emb, ..., hello_emb, world_emb]
    
    After:
        merged: [BOS_emb, img_feat_0, img_feat_1, ..., hello_emb, world_emb]
    """
    # Find positions of image tokens
    image_mask = input_ids == image_token_id
    image_positions = torch.where(image_mask)[0]
    
    if len(image_positions) == 0:
        return text_embeds
    
    # Flatten image features
    flat_image_features = image_features.reshape(-1, image_features.shape[-1])
    
    # Replace embeddings
    merged = text_embeds.clone()
    num_to_replace = min(len(image_positions), flat_image_features.shape[0])
    merged[image_positions[:num_to_replace]] = flat_image_features[:num_to_replace]
    
    return merged

# Demo
hidden_size = 256
num_patches = 4  # Small for demo

# Simulate: [BOS, <img>, <img>, <img>, <img>, hello, world]
input_ids = torch.tensor([1, 32000, 32000, 32000, 32000, 100, 200])
text_embeds = torch.randn(7, hidden_size)  # Random text embeddings
image_features = torch.ones(1, num_patches, hidden_size) * 999  # Distinct image features

merged = merge_multimodal_embeddings(input_ids, text_embeds, image_features, 32000)

print("Token types:")
for i, tid in enumerate(input_ids):
    token_type = "<image>" if tid == 32000 else f"text({tid})"
    is_image = merged[i].mean().item() == 999  # Our sentinel value
    source = "IMAGE_FEATURE" if is_image else "TEXT_EMBEDDING"
    print(f"  Position {i}: {token_type:10s} -> {source}")

## 9. Handling Multiple Images and Video

In [ ]:
# Multi-image and video handling

print("Multi-Image and Video Support")
print("=" * 60)
print("""
Multi-Image:
    prompt = "<image>\n<image>\nCompare these two images."
    images = [PIL.Image, PIL.Image]
    
    Processing:
    1. Each image processed independently through vision encoder
    2. Each <image> replaced with its own patch tokens
    3. Total tokens = sum of patches per image
    
    Memory: N images x patches_per_image x hidden_size

Video:
    prompt = "<video>\nDescribe what happens."
    video = [frame1, frame2, ..., frameN]
    
    Processing:
    1. Sample K frames from the video
    2. Process each frame through vision encoder
    3. Either:
       a. Concatenate all frame features (simple but long)
       b. Pool/compress frame features (efficient)
    
    Memory: K frames x patches_per_frame x hidden_size

vLLM Multi-Modal Data Format:
    multi_modal_data = {
        "image": [PIL.Image, PIL.Image, ...],   # Multiple images
        # OR
        "video": [frames_tensor],                # Video frames
        # OR
        "audio": [audio_tensor],                 # Audio (for speech models)
    }
""")

# Example: Multi-image token calculation
image_size = 336
patch_size = 14
patches_per_image = (image_size // patch_size) ** 2

for num_images in [1, 2, 4, 8]:
    total_image_tokens = num_images * patches_per_image
    kv_cache_mb = total_image_tokens * 4096 * 2 * 32 * 2 / (1024**2)  # GQA Llama-7B estimate
    print(f"  {num_images} images: {total_image_tokens} tokens, ~{kv_cache_mb:.0f} MB KV cache")

## 10. Adding Image Support to a Text-Only Model

Let's implement the steps to add image support to an existing text-only vLLM model.

In [ ]:
# Step-by-step guide to adding multi-modal support

ADDING_VISION_STEPS = '''
Adding Image Support to a Text-Only Model
==========================================

Step 1: Define the vision encoder
    - Choose: CLIP, SigLIP, or custom
    - Implement as a separate nn.Module
    - Load pre-trained weights

Step 2: Define the projector
    - Maps vision features to text embedding space
    - Usually a 2-layer MLP

Step 3: Modify the model class
    - Add SupportsMultiModal to class hierarchy
    - Add vision_tower and projector as attributes
    - Modify forward() to handle pixel_values

Step 4: Register multi-modal capabilities
    - Use @MULTIMODAL_REGISTRY decorators
    - Define max_image_tokens function
    - Define input_processor function

Step 5: Handle weight loading
    - Load vision encoder weights
    - Load projector weights
    - Load language model weights

Step 6: Test
    - Test text-only (no regression)
    - Test with single image
    - Test with multiple images
    - Compare outputs with HuggingFace
'''

print(ADDING_VISION_STEPS)

In [ ]:
class TextOnlyModel(nn.Module):
    """Original text-only model."""
    
    def __init__(self, config):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=config.hidden_size,
                nhead=config.num_heads,
                dim_feedforward=config.intermediate_size,
                batch_first=True,
            )
            for _ in range(config.num_layers)
        ])
        self.norm = nn.LayerNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
    
    def forward(self, input_ids):
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.lm_head(x)


class VisionEnabledModel(nn.Module):
    """
    The SAME model but with vision support added.
    
    Changes from TextOnlyModel:
    1. Added vision_tower (CLIP encoder)
    2. Added mm_projector (MLP projector)
    3. Modified forward() to handle pixel_values
    4. Added SupportsMultiModal
    """
    
    IMAGE_TOKEN_ID = 32000
    
    def __init__(self, config):
        super().__init__()
        
        # === ORIGINAL text model components ===
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=config.hidden_size,
                nhead=config.num_heads,
                dim_feedforward=config.intermediate_size,
                batch_first=True,
            )
            for _ in range(config.num_layers)
        ])
        self.norm = nn.LayerNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        
        # === NEW: Vision components ===
        self.vision_tower = SimplifiedCLIPVisionModel(
            image_size=getattr(config, 'image_size', 224),
            patch_size=getattr(config, 'patch_size', 14),
            embed_dim=getattr(config, 'vision_dim', config.hidden_size),
            num_layers=2,
            num_heads=4,
        )
        
        self.mm_projector = MultiModalProjector(
            vision_dim=getattr(config, 'vision_dim', config.hidden_size),
            text_dim=config.hidden_size,
            projector_type="mlp2x",
        )
        
        self.config = config
    
    def forward(
        self,
        input_ids: torch.Tensor,
        pixel_values: Optional[torch.Tensor] = None,  # NEW parameter
    ):
        # Get text embeddings
        inputs_embeds = self.embed_tokens(input_ids)
        
        # === NEW: Process and merge image features ===
        if pixel_values is not None:
            # Get image features
            vision_features = self.vision_tower(pixel_values)
            vision_features = vision_features[:, 1:, :]  # Remove CLS
            image_features = self.mm_projector(vision_features)
            
            # Merge with text embeddings
            inputs_embeds = merge_multimodal_embeddings(
                input_ids.squeeze(0) if input_ids.dim() > 1 else input_ids,
                inputs_embeds.squeeze(0) if inputs_embeds.dim() > 2 else inputs_embeds,
                image_features,
                self.IMAGE_TOKEN_ID,
            )
            if inputs_embeds.dim() == 2:
                inputs_embeds = inputs_embeds.unsqueeze(0)
        
        # === UNCHANGED: LLM forward pass ===
        x = inputs_embeds
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.lm_head(x)

# Test both models
class VisionConfig:
    vocab_size = 32001
    hidden_size = 256
    intermediate_size = 512
    num_heads = 4
    num_layers = 2
    image_size = 224
    patch_size = 14
    vision_dim = 256

config = VisionConfig()

# Test text-only
text_model = TextOnlyModel(config)
text_input = torch.randint(1, 32000, (1, 20))
with torch.no_grad():
    text_output = text_model(text_input)
print(f"Text-only model: {text_input.shape} -> {text_output.shape}")

# Test vision-enabled
vision_model = VisionEnabledModel(config)
num_patches = (224 // 14) ** 2
multimodal_input = torch.cat([
    torch.tensor([[1]]),
    torch.full((1, num_patches), 32000),
    torch.randint(1, 32000, (1, 10)),
], dim=1)
pixel_values = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    vision_output = vision_model(multimodal_input, pixel_values)
print(f"Vision model: {multimodal_input.shape} + image -> {vision_output.shape}")

# Text-only mode still works!
with torch.no_grad():
    text_only_output = vision_model(text_input)
print(f"Vision model (text-only): {text_input.shape} -> {text_only_output.shape}")

## 11. Dynamic Resolution and Aspect Ratio

In [ ]:
# LLaVA-NeXT and modern models support dynamic resolution

print("Dynamic Resolution (LLaVA-NeXT / LLaVA-OneVision)")
print("=" * 60)
print("""
Problem: Fixed 224x224 resolution loses detail for high-res images.

Solution: Split high-res images into tiles, process each tile separately.

Example: 672x672 image with patch_size=14, base=336x336
    
    +-----+-----+
    | T1  | T2  |   4 tiles of 336x336
    +-----+-----+   + 1 global thumbnail (resized to 336x336)
    | T3  | T4  |   = 5 images through vision encoder
    +-----+-----+
    
    Tokens per tile: (336/14)^2 = 576
    Total tokens: 5 * 576 = 2880 tokens
    vs. 576 tokens for fixed resolution

Supported grid sizes:
    1x1 (336x336):  576 tokens
    1x2 (336x672):  1152 tokens + thumbnail
    2x1 (672x336):  1152 tokens + thumbnail
    2x2 (672x672):  2304 tokens + thumbnail
    1x3 (336x1008): 1728 tokens + thumbnail
    3x1 (1008x336): 1728 tokens + thumbnail
    ...
""")

def calculate_dynamic_tokens(image_height, image_width, patch_size=14, base_size=336, max_tiles=6):
    """Calculate number of tokens for dynamic resolution."""
    patches_per_tile = (base_size // patch_size) ** 2
    
    # Determine best grid
    aspect_ratio = image_width / image_height
    best_grid = (1, 1)
    best_waste = float('inf')
    
    for h in range(1, max_tiles + 1):
        for w in range(1, max_tiles + 1):
            if h * w > max_tiles:
                continue
            grid_ratio = w / h
            waste = abs(grid_ratio - aspect_ratio)
            if waste < best_waste:
                best_waste = waste
                best_grid = (h, w)
    
    num_tiles = best_grid[0] * best_grid[1]
    # +1 for global thumbnail
    total_tokens = (num_tiles + 1) * patches_per_tile
    
    return best_grid, num_tiles, total_tokens

# Examples
test_images = [
    (336, 336, "Square"),
    (672, 672, "2x2 Square"),
    (336, 672, "1x2 Wide"),
    (1008, 336, "3x1 Tall"),
    (1920, 1080, "Full HD"),
]

print(f"\n{'Image Size':>15} {'Grid':>8} {'Tiles':>6} {'Tokens':>8} {'Description'}")
print("-" * 60)
for h, w, desc in test_images:
    grid, tiles, tokens = calculate_dynamic_tokens(h, w)
    print(f"{h}x{w:>4} {str(grid):>8} {tiles:>6} {tokens:>8} {desc}")

## 12. Weight Loading for Multi-Modal Models

In [ ]:
# Weight loading for multi-modal models

WEIGHT_LOADING_CODE = '''
def load_weights(self, weights: Iterable[Tuple[str, torch.Tensor]]):
    """Load weights for a multi-modal model."""
    
    # Weight name prefixes
    # HuggingFace LLaVA weights have these prefixes:
    #   vision_tower.* -> Vision encoder weights
    #   multi_modal_projector.* -> Projector weights
    #   language_model.* -> LLM weights
    
    stacked_params = [
        ("qkv_proj", "q_proj", "q"),
        ("qkv_proj", "k_proj", "k"),
        ("qkv_proj", "v_proj", "v"),
        ("gate_up_proj", "gate_proj", 0),
        ("gate_up_proj", "up_proj", 1),
    ]
    
    params_dict = dict(self.named_parameters())
    loaded = set()
    
    for name, weight in weights:
        # Skip vision encoder internal weights we don't need
        if "vision_tower" in name and "position_ids" in name:
            continue
        
        # Handle language model stacked params
        if "language_model" in name:
            for (vllm_name, hf_name, shard_id) in stacked_params:
                if hf_name in name:
                    name = name.replace(hf_name, vllm_name)
                    # ... load with shard_id
                    break
        
        # Direct copy for vision tower and projector
        if name in params_dict:
            params_dict[name].data.copy_(weight)
            loaded.add(name)
    
    return loaded
'''

print(WEIGHT_LOADING_CODE)

# Show typical weight names
print("\nTypical LLaVA weight names:")
weight_names = [
    "vision_tower.vision_model.embeddings.patch_embedding.weight",
    "vision_tower.vision_model.encoder.layers.0.self_attn.q_proj.weight",
    "vision_tower.vision_model.encoder.layers.0.mlp.fc1.weight",
    "multi_modal_projector.linear_1.weight",
    "multi_modal_projector.linear_2.weight",
    "language_model.model.embed_tokens.weight",
    "language_model.model.layers.0.self_attn.q_proj.weight",
    "language_model.model.layers.0.mlp.gate_proj.weight",
    "language_model.lm_head.weight",
]
for name in weight_names:
    component = name.split(".")[0]
    print(f"  [{component:30s}] {name}")

## 13. Exercises

### Exercise 1: Implement a Perceiver Resampler
Implement a perceiver resampler that reduces 256 patch tokens to 64 query tokens using cross-attention.

### Exercise 2: Add Video Support
Extend the `VisionEnabledModel` to support video input (multiple frames). Handle frame sampling and feature aggregation.

### Exercise 3: Multi-Image Model
Modify the `merge_multimodal_embeddings` function to handle multiple images in a single prompt, where each `<image>` placeholder corresponds to a different image.

In [ ]:
# Exercise 1 starter code: Perceiver Resampler

class PerceiverResampler(nn.Module):
    """
    Perceiver Resampler: reduces variable-length vision features
    to a fixed number of tokens.
    
    Input:  [batch, num_patches, vision_dim]  (variable num_patches)
    Output: [batch, num_queries, text_dim]     (fixed num_queries)
    """
    
    def __init__(self, vision_dim, text_dim, num_queries=64, num_heads=8, num_layers=2):
        super().__init__()
        self.num_queries = num_queries
        
        # Learnable query tokens
        self.queries = nn.Parameter(torch.randn(1, num_queries, text_dim) * 0.02)
        
        # Project vision features to text dim
        self.input_proj = nn.Linear(vision_dim, text_dim)
        
        # Cross-attention layers
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'cross_attn': nn.MultiheadAttention(text_dim, num_heads, batch_first=True),
                'norm1': nn.LayerNorm(text_dim),
                'ff': nn.Sequential(
                    nn.Linear(text_dim, text_dim * 4),
                    nn.GELU(),
                    nn.Linear(text_dim * 4, text_dim),
                ),
                'norm2': nn.LayerNorm(text_dim),
            })
            for _ in range(num_layers)
        ])
    
    def forward(self, vision_features: torch.Tensor) -> torch.Tensor:
        batch_size = vision_features.shape[0]
        
        # Project vision features
        kv = self.input_proj(vision_features)
        
        # Initialize queries
        queries = self.queries.expand(batch_size, -1, -1)
        
        # Process through cross-attention layers
        for layer in self.layers:
            # Cross-attention: queries attend to vision features
            h = layer['norm1'](queries)
            h, _ = layer['cross_attn'](h, kv, kv)
            queries = queries + h
            
            # Feed-forward
            h = layer['norm2'](queries)
            queries = queries + layer['ff'](h)
        
        return queries

# Test
resampler = PerceiverResampler(vision_dim=256, text_dim=512, num_queries=64)
vision_feats = torch.randn(2, 257, 256)  # 2 images, 257 patches each
with torch.no_grad():
    output = resampler(vision_feats)

print(f"Perceiver Resampler:")
print(f"  Input:  {vision_feats.shape} (variable patches)")
print(f"  Output: {output.shape} (fixed {resampler.num_queries} queries)")
print(f"  Compression: {vision_feats.shape[1]} -> {output.shape[1]} tokens ({vision_feats.shape[1]/output.shape[1]:.1f}x)")

## Summary

In this notebook, we covered:

1. **Architecture Patterns**: Fused input (LLaVA), cross-attention (Flamingo), early fusion (Fuyu)
2. **Vision Encoder**: CLIP ViT with patch embeddings and transformer blocks
3. **Projector**: Linear, MLP, and Perceiver Resampler designs
4. **Image Processing**: Preprocessing pipeline with resize and normalization
5. **vLLM Integration**: SupportsMultiModal, MULTIMODAL_REGISTRY, input_processor
6. **Embedding Merge**: Replace `<image>` placeholder embeddings with vision features
7. **Dynamic Resolution**: Tile-based processing for high-resolution images
8. **Weight Loading**: Handling vision + projector + LLM weights

### Key Takeaway
The core idea is simple: convert images to token embeddings (via vision encoder + projector), then treat them like text tokens. The LLM processes both modalities uniformly.

### Next: Chapter 8.5 -- Model Registration in SGLang